In [ ]:
# imports and settings

import os
import time
import pickle
import warnings
import pandas as pd
import networkx as nx

import numpy as np
from numpy import linalg as LA
from numpy import histogram2d

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.metrics import auc

import utils as ut
%load_ext autoreload
%autoreload 2

# do not show warnings
warnings.filterwarnings("ignore")

print("Imports complete.")

## Detector Sanity Check

In [ ]:
# simulation parameters
fs = 128000  # sampling frequency
f0 = 640  # signal frequency
duration = 60  # seconds
snr_values = [0, -2, -4, -6, -8]  # from 0 dB to -8 dB

# detector args
nperseg = 32000
nfft = None # 2*nperseg
noverlap = 0.5
window = 'hanning'
dc = 20
crop_freq = 8000
norm_size = 5
quantization_bins = 3
welch_threshold = 0.034
s2g_threshold_laplacian = 0.24
s2g_threshold_wasserstein = 16

raw_signal = ut.simulate_raw_signal(f0, fs, duration)
F, T, raw_Sxx, raw_phasogram = ut.calc_spectrogram(raw_signal, fs, nfft=nfft, nperseg=nperseg, percent_overlap=noverlap/nperseg, window=window, remove_dc=dc, crop_freq=crop_freq)

welch_detector = ut.WelchDetector(fs=fs, nperseg=nperseg, overlap=noverlap, nfft=nfft, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
s2g_detector_laplacian = ut.S2GDetector(fs=fs, nperseg=nperseg, overlap=noverlap, nfft=nfft, window=window, dc=dc, crop_freq=crop_freq, quantization_levels=quantization_bins, mode="laplacian")
s2g_detector_wasserstein = ut.S2GDetector(fs=fs, nperseg=nperseg, overlap=noverlap, nfft=nfft, window=window, dc=dc, crop_freq=crop_freq, quantization_levels=quantization_bins, mode="wasserstein")

fig = make_subplots(rows=4, cols=len(snr_values), shared_xaxes=True, column_titles=[f"{snr} dB" for snr in snr_values])
for i, snr in enumerate(snr_values, start=1):
    rx = ut.add_noise_to_signal(raw_signal, snr_db=snr, fs=fs, signal_bw=1, noise_type='white')

    _F, _T, rx_Sxx, _phasogram = ut.calc_spectrogram(rx, fs, nperseg=nperseg, percent_overlap=noverlap, window=window, remove_dc=dc, crop_freq=crop_freq)
    fig.add_trace(go.Heatmap(z=rx_Sxx.T, y=_T, x=_F, colorscale='Viridis', showlegend=False, showscale=False), row=1, col=i)

    rx_pxx = ut.calc_welch_from_spectrogram(rx_Sxx, normalization_window_size=norm_size)
    _, welch_detections, _ = welch_detector.detect(rx, threshold=welch_threshold)
    fig.add_trace(go.Scatter(y=rx_pxx, x=_F, showlegend=False, line=dict(color='blue')), row=2, col=i)
    # if len(welch_detections) > 0:
    #     fig.add_trace(go.Scatter(y=rx_pxx[welch_detections], x=_F[welch_detections], mode='markers', marker=dict(color='red', size=10), showlegend=False), row=2, col=i)

    _, s2g_detections_l, (K_l, K_f_l) = s2g_detector_laplacian.detect(rx, threshold=s2g_threshold_laplacian)
    fig.add_trace(go.Scatter(y=K_l, x=K_f_l, showlegend=False, line=dict(color='purple')), row=3, col=i)
    # if len(s2g_detections_l) > 0:
    #     fig.add_trace(go.Scatter(y=K_l[s2g_detections_l], x=K_f_l[s2g_detections_l], mode='markers', marker=dict(color='red', size=10), showlegend=False), row=3, col=i)

    _, s2g_detections_w, (K_w, K_f_w) = s2g_detector_wasserstein.detect(rx, threshold=s2g_threshold_wasserstein)
    fig.add_trace(go.Scatter(y=K_w, x=K_f_w, showlegend=False, line=dict(color='green')), row=4, col=i)
    # if len(s2g_detections_w) > 0:
    #     fig.add_trace(go.Scatter(y=K_w[s2g_detections_w], x=K_f_w[s2g_detections_w], mode='markers', marker=dict(color='red', size=10), showlegend=False), row=4, col=i)


fig.update_layout(height=800, width=300*len(snr_values), title_text=f"fs={fs} | f0={f0} | fftsize={nperseg} | quantization_bins={quantization_bins} | welch_threshold={welch_threshold} | s2g_laplacian_threshold={s2g_threshold_laplacian} | s2g_wasserstein_threshold={s2g_threshold_wasserstein}")
fig.show(renderer="browser")

---

## Simulation Sanity Check

In [ ]:
# simulation settings
fs = 128000  # sampling frequency
f0 = 640  # signal frequency
duration = 60  # seconds

quantization_range = [2, 3, 4, 5, 6, 10, 15, 20, 25]

search_precision = 10
welch_thresholds = np.linspace(0.01, 0.1, search_precision)
s2g_laplacian_thresholds = np.linspace(0.1, 0.99, search_precision)
s2g_wasserstein_thresholds = np.linspace(0.01, 0.2, search_precision)

# detectors
welch_detector = ut.WelchDetector(fs, nperseg)
s2g_detector_laplacian = ut.S2GDetector(fs, nperseg, mode="laplacian")
s2g_detector_wasserstein = ut.S2GDetector(fs, nperseg, mode="wasserstein")

In [ ]:
# sanity check - run one simulation and plot results
snr = -6
tolerance = 5  # Hz, for matching detections to true signal frequency
raw_signal = ut.simulate_raw_signal(f0, fs, duration)
rx = ut.add_noise_to_signal(raw_signal, snr_db=snr, fs=fs, signal_bw=1, noise_type='white')
T, F, rx_Sxx, rx_phasogram = ut.calc_spectrogram(rx, fs, nperseg=nperseg, percent_overlap=noverlap, window=window, remove_dc=dc, crop_freq=crop_freq)

welch_results_all_configurations = []
s2g_laplacian_results_all_configurations = []
s2g_wasserstein_results_all_configurations = []

for fft_size in fft_range:
    welch_detector.set_nperseg(fft_size)
    s2g_detector_laplacian.set_nperseg(fft_size)
    s2g_detector_wasserstein.set_nperseg(fft_size)

    for welch_threshold in welch_thresholds:
        _, welch_detections, (pxx, F_pxx) = welch_detector.detect(rx, threshold=welch_threshold)
        welch_is_detected = any(abs(F_pxx[welch_detections] - f0) <= tolerance)
        welch_false_alarm = any(abs(F_pxx[welch_detections] - f0) > tolerance)
        welch_results_all_configurations.append({"detector": "welch", "fft_size": fft_size, "welch_threshold": welch_threshold, "welch_is_detected": welch_is_detected, "welch_false_alarm": welch_false_alarm})

    for s2g_threshold in s2g_laplacian_thresholds:
        _, s2g_detections, (K, F_s2g) = s2g_detector_laplacian.detect(rx, threshold=s2g_threshold)
        s2g_is_detected = any(abs(F_s2g[s2g_detections] - f0) <= tolerance)
        s2g_false_alarm = any(abs(F_s2g[s2g_detections] - f0) > tolerance)
        s2g_laplacian_results_all_configurations.append({"detector": "s2g_laplacian", "fft_size": fft_size, "s2g_threshold": s2g_threshold, "s2g_is_detected": s2g_is_detected, "s2g_false_alarm": s2g_false_alarm})

    for s2g_threshold in s2g_wasserstein_thresholds:
        _, s2g_detections, (K, F_s2g) = s2g_detector_wasserstein.detect(rx, threshold=s2g_threshold)
        s2g_is_detected = any(abs(F_s2g[s2g_detections] - f0) <= tolerance)
        s2g_false_alarm = any(abs(F_s2g[s2g_detections] - f0) > tolerance)
        s2g_wasserstein_results_all_configurations.append({"detector": "s2g_wasserstein", "fft_size": fft_size, "s2g_threshold": s2g_threshold, "s2g_is_detected": s2g_is_detected, "s2g_false_alarm": s2g_false_alarm})

print(f"welch results: {welch_results_all_configurations}")
print(f"s2g laplacian results: {s2g_laplacian_results_all_configurations}")
print(f"s2g wasserstein results: {s2g_wasserstein_results_all_configurations}")

---

## Simulation Hyperparameter Search

In [ ]:
# simulation settings
fs = 8000  # sampling frequency
f0 = 640  # signal frequency
duration = 60  # seconds

quantization_range = [2, 3, 4, 5, 10, 15, 20]
fft_range = [256, 512, 1024, 2048, 4096]

search_precision = 20
welch_thresholds = np.linspace(0.01, 0.1, search_precision)
s2g_laplacian_thresholds = np.linspace(0.1, 0.99, search_precision)
s2g_wasserstein_thresholds = np.linspace(0.01, 0.2, search_precision)

## MAIN SIMULATION CODE
num_iterations = 100
snr_values = [0, -2, -4, -6, -8, -10]  # from 0 dB to -10 dB
raw_signal = ut.simulate_raw_signal(f0, fs, duration)
tolerance = 5  # Hz, for matching detections to true signal frequency
T, F, raw_Sxx, raw_phasogram = ut.calc_spectrogram(raw_signal, fs, nperseg=nperseg, percent_overlap=noverlap, window=window, remove_dc=dc, crop_freq=crop_freq)

In [ ]:

print("Starting simulation for Welch Detector...")
# for each detector: welch, s2g laplacian, s2g wasserstein
# iterate over configurations: fft_size, quantization, threshold
# for each configuration, run num_iterations simulations at different SNRs,
# for each condiguration and snr, calculate detection probability and FAR, 
# store results (detection, false alarm) for each configuration and snr as a dictionary.

In [ ]:

welch_results = dict()
print("Evaluating hyperparameters for Welch Detector...")
for fft_size in fft_range:
    welch_detector = ut.WelchDetector(fs, fft_size)
    for threshold in welch_thresholds:
        for snr in snr_values:
            welch_detections = []
            welch_false_alarms = []
            for i in range(num_iterations):
                print(f"Welch Detector - FFT Size: {fft_size}, Threshold: {threshold:.4f}, SNR: {snr} dB, Iteration: {i+1}/{num_iterations}")
                rx = ut.add_noise_to_signal(raw_signal, snr_db=snr, fs=fs, signal_bw=1, noise_type='white')
                _, detections, (pxx, F_pxx) = welch_detector.detect(rx, threshold=threshold)
                is_detected = any(abs(F_pxx[detections] - f0) <= tolerance)
                false_alarm = any(abs(F_pxx[detections] - f0) > tolerance)
                welch_detections.append(is_detected)
                welch_false_alarms.append(false_alarm)
            welch_results[(fft_size, snr, threshold)] = {"detection_probability": np.mean(welch_detections), "false_alarm_rate": np.mean(welch_false_alarms)}
with open("hyperparameter_eval_welch_results.pkl", "wb") as f:
    pickle.dump(welch_results, f)

In [ ]:

s2g_laplacian_results = dict()
print("Evaluating hyperparameters for S2G Laplacian Detector...")
for fft_size in fft_range:
    for quantization in quantization_range:
        s2g_detector_laplacian = ut.S2GDetector(fs, fft_size, mode="laplacian", quantization_levels=quantization)
        for threshold in s2g_laplacian_thresholds:
            for snr in snr_values:
                s2g_detections = []
                s2g_false_alarms = []
                for i in range(num_iterations):
                    print(f"S2G Laplacian Detector - FFT Size: {fft_size}, Threshold: {threshold:.4f}, SNR: {snr} dB, Iteration: {i+1}/{num_iterations}")
                    rx = ut.add_noise_to_signal(raw_signal, snr_db=snr, fs=fs, signal_bw=1, noise_type='white')
                    _, detections, (K, F_s2g) = s2g_detector_laplacian.detect(rx, threshold=threshold)
                    is_detected = any(abs(F_s2g[detections] - f0) <= tolerance)
                    false_alarm = any(abs(F_s2g[detections] - f0) > tolerance)
                    s2g_detections.append(is_detected)
                    s2g_false_alarms.append(false_alarm)
                s2g_laplacian_results[(fft_size, quantization, snr, threshold)] = {"detection_probability": np.mean(s2g_detections), "false_alarm_rate": np.mean(s2g_false_alarms)}
with open("hyperparameter_eval_s2g_laplacian_results.pkl", "wb") as f:
    pickle.dump(s2g_laplacian_results, f)

In [ ]:

s2g_wasserstein_results = dict()
print("Evaluating hyperparameters for S2G Wasserstein Detector...")
for fft_size in fft_range:
    for quantization in quantization_range:
        s2g_detector_wasserstein = ut.S2GDetector(fs, fft_size, mode="wasserstein", quantization_levels=quantization)
        for threshold in s2g_wasserstein_thresholds:
            for snr in snr_values:
                s2g_detections = []
                s2g_false_alarms = []
                for i in range(num_iterations):
                    print(f"S2G Wasserstein Detector - FFT Size: {fft_size}, Threshold: {threshold:.4f}, SNR: {snr} dB, Iteration: {i+1}/{num_iterations}")
                    rx = ut.add_noise_to_signal(raw_signal, snr_db=snr, fs=fs, signal_bw=1, noise_type='white')
                    _, detections, (K, F_s2g) = s2g_detector_wasserstein.detect(rx, threshold=threshold)
                    is_detected = any(abs(F_s2g[detections] - f0) <= tolerance)
                    false_alarm = any(abs(F_s2g[detections] - f0) > tolerance)
                    s2g_detections.append(is_detected)
                    s2g_false_alarms.append(false_alarm)
                s2g_wasserstein_results[(fft_size, quantization, snr, threshold)] = {"detection_probability": np.mean(s2g_detections), "false_alarm_rate": np.mean(s2g_false_alarms)}
with open("hyperparameter_eval_s2g_wasserstein_results.pkl", "wb") as f:
    pickle.dump(s2g_wasserstein_results, f)

In [ ]:
print("Simulation complete. Results saved to pickle files.")

---

## Processing

In [ ]:
with open("hyperparameter_eval_welch_results.pkl", "rb") as f:
    welch_results = pickle.load(f)

welch_roc = dict()
for fft_size in fft_range:
    for snr in snr_values:
        far = []
        pd = []
        for threshold in welch_thresholds:
            far.append(welch_results[(fft_size, snr, threshold)]["false_alarm_rate"])
            pd.append(welch_results[(fft_size, snr, threshold)]["detection_probability"])
        
        sorted_indices = np.argsort(far)
        far_sorted = np.array(far)[sorted_indices]
        pd_sorted = np.array(pd)[sorted_indices]
        auc_score = auc(far_sorted, pd_sorted)
        welch_roc[(fft_size, snr)] = {"auc": auc_score, "fa": far, "pd": pd}
        # print(f"FFT Size: {fft_size}, SNR: {snr} dB, AUC: {auc_score:.4f}")

with open("hyperparameter_eval_welch_results_with_auc.pkl", "wb") as f:
    pickle.dump(welch_roc, f)


In [ ]:
with open("hyperparameter_eval_s2g_laplacian_results.pkl", "rb") as f:
    s2g_laplacian_results = pickle.load(f)

s2g_laplacian_roc = dict()
for fft_size in fft_range:
    for quantization in quantization_range:
        for snr in snr_values:
            far = []
            pd = []
            for threshold in s2g_laplacian_thresholds:
                far.append(s2g_laplacian_results[(fft_size, quantization, snr, threshold)]["false_alarm_rate"])
                pd.append(s2g_laplacian_results[(fft_size, quantization, snr, threshold)]["detection_probability"])
            
            sorted_indices = np.argsort(far)
            far_sorted = np.array(far)[sorted_indices]
            pd_sorted = np.array(pd)[sorted_indices]
            auc_score = auc(far_sorted, pd_sorted)
            s2g_laplacian_roc[(fft_size, quantization, snr)] = {"auc": auc_score, "fa": far, "pd": pd}
            # print(f"FFT Size: {fft_size}, Quantization: {quantization}, SNR: {snr} dB, AUC: {auc_score:.4f}")

with open("hyperparameter_eval_s2g_laplacian_results_with_auc.pkl", "wb") as f:
    pickle.dump(s2g_laplacian_roc, f)

In [ ]:
with open("hyperparameter_eval_s2g_wasserstein_results.pkl", "rb") as f:
    s2g_wasserstein_results = pickle.load(f)

s2g_wasserstein_roc = dict()
for fft_size in fft_range:
    for quantization in quantization_range:
        for snr in snr_values:
            far = []
            pd = []
            for threshold in s2g_wasserstein_thresholds:
                far.append(s2g_wasserstein_results[(fft_size, quantization, snr, threshold)]["false_alarm_rate"])
                pd.append(s2g_wasserstein_results[(fft_size, quantization, snr, threshold)]["detection_probability"])
            if far and pd:
                auc_score = auc(far, pd)
                s2g_wasserstein_roc[(fft_size, quantization, snr)] = {"auc": auc_score, "fa": far, "pd": pd}
                print(f"FFT Size: {fft_size}, Quantization: {quantization}, SNR: {snr} dB, AUC: {auc_score:.4f}")

with open("hyperparameter_eval_s2g_wasserstein_results_with_auc.pkl", "wb") as f:
    pickle.dump(s2g_wasserstein_roc, f)

---

# Analysis

In [ ]:
# plot AUC vs SNR for each detector, with different lines for different fft sizes (and quantization levels for s2g)
from collections import defaultdict

def plot_auc_vs_snr(results2):
    """
    Plots AUC over SNR for different detection algorithm configurations.
    
    Args:
        results2 (dict): Dictionary with keys (fft_size, quantization, snr) 
                         and values (auc, pd_over_thresholds, far_over_thresholds).
    """
    # Group the data by configuration (fft_size, quantization)
    config_data = defaultdict(lambda: {'snr': [], 'auc': []})
    
    for (fft_size, quantization, snr), (auc, _, _) in results2.items():
        config = (fft_size, quantization)
        config_data[config]['snr'].append(snr)
        config_data[config]['auc'].append(auc)

    fig = go.Figure()

    # Create a line for each configuration
    for (fft_size, quantization), data in config_data.items():
        # Sort by SNR to ensure the lines are drawn left-to-right correctly
        sorted_pairs = sorted(zip(data['snr'], data['auc']))
        snr_sorted = [pair[0] for pair in sorted_pairs]
        auc_sorted = [pair[1] for pair in sorted_pairs]

        # Define custom hover text
        custom_hover = (
            f"<b>FFT Size:</b> {fft_size}<br>"
            f"<b>Quantization:</b> {quantization}<br>"
            "<b>SNR:</b> %{x}<br>"
            "<b>AUC:</b> %{y:.4f}"
            "<extra></extra>" # Removes the default secondary trace label box
        )

        fig.add_trace(go.Scatter(
            x=snr_sorted,
            y=auc_sorted,
            mode='lines+markers',
            name=f"FFT: {fft_size} | Quant: {quantization}",
            hovertemplate=custom_hover
        ))

    # Update the layout for better readability
    fig.update_layout(
        title="Detection Algorithm Performance: AUC vs. SNR",
        xaxis_title="Signal-to-Noise Ratio (SNR)",
        yaxis_title="Area Under the Curve (AUC)",
        legend_title="Configurations",
        hovermode="closest",
        template="plotly_white"
    )

    fig.show(renderer="browser")

In [ ]:
with open("hyperparameter_eval_s2g_laplacian_results_with_auc.pkl", "rb") as f:
    s2g_laplacian_roc = pickle.load(f)
plot_auc_vs_snr(s2g_laplacian_roc)

In [ ]:
with open('hyperparameter_eval_s2g_laplacian_results.pkl', "rb") as f:
    s2g_laplacian_results = pickle.load(f)

# create pandas dataframe from results
res_df = pd.DataFrame(s2g_laplacian_results).T.reset_index()
res_df.columns = ["fft_size", "quantization", "snr", "threshold", "detection_probability", "false_alarm_rate"]
res_df.iloc[550:555]